### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)
from sklearn.model_selection import GridSearchCV, train_test_split

### Load Train & Test

In [2]:
X_train, X_val, y_train, y_val = joblib.load("../models/processed_train_val.joblib")
X_test = joblib.load("../models/processed_test.joblib")
preprocessor = joblib.load("../models/preprocessor.joblib")

train_df = pd.read_csv("../data/train_updated.csv")
X_raw_full = train_df.drop("RiskFlag", axis=1)
y_raw_full = train_df["RiskFlag"]



### Evaluation

In [3]:
def evaluate(name, y_true, y_pred, y_prob=None):
    print(f"\n================ {name} ================")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall   :", recall_score(y_true, y_pred))
    print("F1-score :", f1_score(y_true, y_pred))
    if y_prob is not None:
        print("ROC-AUC  :", roc_auc_score(y_true, y_prob))
    print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("\nClassification Report:\n", classification_report(y_true, y_pred))

### Baseline Full Dataset

In [4]:
lr_base_full = LogisticRegression(
    solver="liblinear",
    max_iter=600,
    class_weight="balanced"
)

lr_base_full.fit(X_train, y_train)

pred_f = lr_base_full.predict(X_val)
prob_f = lr_base_full.predict_proba(X_val)[:,1]

evaluate("LR Baseline — FULL", y_val, pred_f, prob_f)

f1_full_base = f1_score(y_val, pred_f)



================ LR Baseline — FULL ================
Accuracy : 0.6712355590366164
Precision: 0.2158428805237316
Recall   : 0.693959166491265
F1-score : 0.3292719464695895
ROC-AUC  : 0.7465545646685043

Confusion Matrix:
 [[24127 11978]
 [ 1454  3297]]

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.67      0.78     36105
           1       0.22      0.69      0.33      4751

    accuracy                           0.67     40856
   macro avg       0.58      0.68      0.56     40856
weighted avg       0.86      0.67      0.73     40856



### Full Dataset: Hyperparameter Tuning

In [5]:
param_grid = {
    "C": [0.01, 0.1, 1, 5, 10],
    "penalty": ["l1", "l2"]
}

lr_grid_full = GridSearchCV(
    estimator=LogisticRegression(
        solver="liblinear",
        max_iter=2000,
        class_weight="balanced"
    ),
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)

lr_grid_full.fit(X_train, y_train)
print("Best FULL params:", lr_grid_full.best_params_)

lr_best_full = lr_grid_full.best_estimator_
pred_f_tuned = lr_best_full.predict(X_val)
prob_f_tuned = lr_best_full.predict_proba(X_val)[:,1]

evaluate("LR Tuned — FULL", y_val, pred_f_tuned, prob_f_tuned)
f1_full_tuned = f1_score(y_val, pred_f_tuned)


Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best FULL params: {'C': 5, 'penalty': 'l1'}

================ LR Tuned — FULL ================
Accuracy : 0.6712355590366164
Precision: 0.2158428805237316
Recall   : 0.693959166491265
F1-score : 0.3292719464695895
ROC-AUC  : 0.7465547920275445

Confusion Matrix:
 [[24127 11978]
 [ 1454  3297]]

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.67      0.78     36105
           1       0.22      0.69      0.33      4751

    accuracy                           0.67     40856
   macro avg       0.58      0.68      0.56     40856
weighted avg       0.86      0.67      0.73     40856



### Create 20% dataset

In [6]:
df_20, _ = train_df.sample(frac=0.2, random_state=42).align(train_df, join='left', axis=1)
X_20 = preprocessor.transform(df_20.drop("RiskFlag", axis=1))
y_20 = df_20["RiskFlag"]
X_train_20, X_val_20, y_train_20, y_val_20 = train_test_split(
    X_20, y_20,
    test_size=0.2,
    random_state=42,
    stratify=y_20
)


### Baseline LR -20%dataset

In [7]:
lr_base_20 = LogisticRegression(
    solver="liblinear",
    max_iter=600,
    class_weight="balanced"
)

lr_base_20.fit(X_train_20, y_train_20)

pred_20 = lr_base_20.predict(X_val_20)
prob_20 = lr_base_20.predict_proba(X_val_20)[:,1]

evaluate("LR Baseline — 20%", y_val_20, pred_20, prob_20)

f1_20_base = f1_score(y_val_20, pred_20)



================ LR Baseline — 20% ================
Accuracy : 0.6759270591114919
Precision: 0.22031094938802515
Recall   : 0.6959247648902821
F1-score : 0.33467336683417087
ROC-AUC  : 0.7555668633410189

Confusion Matrix:
 [[4857 2357]
 [ 291  666]]

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.67      0.79      7214
           1       0.22      0.70      0.33       957

    accuracy                           0.68      8171
   macro avg       0.58      0.68      0.56      8171
weighted avg       0.86      0.68      0.73      8171



### Hyperparameter Tuning -20%dataset

In [8]:
lr_grid_20 = GridSearchCV(
    estimator=LogisticRegression(
        solver="liblinear",
        max_iter=2000,
        class_weight="balanced"
    ),
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)

lr_grid_20.fit(X_train_20, y_train_20)
print("Best 20% params:", lr_grid_20.best_params_)

lr_best_20 = lr_grid_20.best_estimator_
pred_20_tuned = lr_best_20.predict(X_val_20)
prob_20_tuned = lr_best_20.predict_proba(X_val_20)[:,1]

evaluate("LR Tuned — 20%", y_val_20, pred_20_tuned, prob_20_tuned)

f1_20_tuned = f1_score(y_val_20, pred_20_tuned)



Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best 20% params: {'C': 0.1, 'penalty': 'l1'}

================ LR Tuned — 20% ================
Accuracy : 0.6767837473993391
Precision: 0.22119205298013245
Recall   : 0.6980146290491118
F1-score : 0.3359316067387478
ROC-AUC  : 0.7553862381257389

Confusion Matrix:
 [[4862 2352]
 [ 289  668]]

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.67      0.79      7214
           1       0.22      0.70      0.34       957

    accuracy                           0.68      8171
   macro avg       0.58      0.69      0.56      8171
weighted avg       0.86      0.68      0.73      8171



### Comparison Table

In [9]:
comparison = pd.DataFrame({
    "Variant": [
        "Full Baseline",
        "Full Tuned",
        "20% Baseline",
        "20% Tuned"
    ],
    "F1 Score": [
        f1_full_base,
        f1_full_tuned,
        f1_20_base,
        f1_20_tuned
    ]
})

print("\n========== Logistic Regression Comparison ==========\n")
print(comparison)


========== Logistic Regression Comparison ==========

         Variant  F1 Score
0  Full Baseline  0.329272
1     Full Tuned  0.329272
2   20% Baseline  0.334673
3      20% Tuned  0.335932


### Generate Submissions

In [11]:
X_full_pre = preprocessor.transform(X_raw_full)
final_lr_full = lr_best_full
final_lr_full.fit(X_full_pre, y_raw_full)
lr_test_full = final_lr_full.predict(X_test)
test_df = pd.read_csv("../data/test_updated.csv")
sub_full = pd.DataFrame({
    "ProfileID": test_df["ProfileID"],
    "RiskFlag": lr_test_full
})

sub_full.to_csv("../submissions/submission_lr_full.csv", index=False)
print("submission_lr_full.csv saved")
X_20_pre = preprocessor.transform(df_20.drop("RiskFlag", axis=1))
final_lr_20 = lr_best_20
final_lr_20.fit(X_20_pre, df_20["RiskFlag"])
lr_test_20 = final_lr_20.predict(X_test)

sub20 = pd.DataFrame({
    "ProfileID": test_df["ProfileID"],
    "RiskFlag": lr_test_20
})
sub20.to_csv("../submissions/submission_lr_20.csv", index=False)
print("submission_lr_20.csv saved")


submission_lr_full.csv saved
submission_lr_20.csv saved
